# LDM named experiments and Hugging Face release

Run setup cells once per Colab session. Define several package-typed presets, choose one, then use **Start**, **Inspect**, or **Resume** separately. New runs are stored on Drive under unique IDs. The currently existing `runs/with_subd` folder is left untouched. Only publish after reviewing a trained checkpoint.


## 1. Code and dependencies

Add `GITHUB_TOKEN` to Colab Secrets, allow notebook access, and ensure the token can read `appodoom/ldm`.

In [ ]:
import base64
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

project = Path("/content/ldm")
token = userdata.get("GITHUB_TOKEN")
assert token, "Add GITHUB_TOKEN to Colab Secrets and allow notebook access"
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
})
if (project / ".git").exists():
    subprocess.run(["git", "-C", str(project), "pull", "--ff-only", "origin", "main"],
                   env=git_env, check=True)
elif project.exists():
    raise RuntimeError("/content/ldm exists but is not the Git checkout")
else:
    subprocess.run(["git", "clone", "https://github.com/appodoom/ldm.git", str(project)],
                   env=git_env, check=True)
code_commit = subprocess.check_output(
    ["git", "-C", str(project), "rev-parse", "HEAD"], text=True).strip()
print("Code commit:", code_commit)


In [ ]:
os.chdir(project)
print("Python:", sys.version.split()[0])
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               check=True)


## 2. Pin the source corpus and mount Drive

This notebook uses the pinned five-line normal and decorated data. Run directories on Drive survive runtime restarts.

In [ ]:
from huggingface_hub import notebook_login, snapshot_download

notebook_login()
data_root = Path(snapshot_download(
    repo_id="Appodoom/ldm-data", repo_type="dataset",
    revision="34fb2449a25472e856a4db0aa85e280e9a82a8b2",
    allow_patterns="five-line/**", local_dir=project / ".derbake",
))
ordinary = data_root / "five-line" / "normal"
decorated = data_root / "five-line" / "decorated"
assert len(list(ordinary.glob("*.derbake"))) == 116
assert len(list(decorated.glob("*.derbake"))) == 116
print("Paired files:", len(list(ordinary.glob("*.derbake"))))


In [ ]:
from google.colab import drive
import torch, json
drive.mount('/content/drive')
storage_root = Path('/content/drive/MyDrive/LDM')
runs_root = storage_root / 'runs'
runs_root.mkdir(parents=True, exist_ok=True)
cache_root = project / 'datasets'
print('Runs:', runs_root, '| Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## 3. Named run configurations

Add or edit presets here. Choices such as variant, attention, activation and position encoding come from the GitHub package; numerical settings live in each preset. A config becomes immutable once its run starts.

In [ ]:
from ldm.experiment_config import (
    Activation, AttentionLayout, DatasetVariant, PositionEncoding,
    DataConfig, ModelConfig, TrainConfig, EvalConfig, RunConfig, StructuredStreams,
)

RUN_CONFIGS = {
    'with_subd_small': RunConfig(
        name='with_subd_small',
        data=DataConfig(variant=DatasetVariant.WITH_SUBD, context_size=1024,
                        stride=512, max_beats_per_cycle=16, include_decorated=True,
                        decorated_weight=1.0),
        model=ModelConfig(layers=4, heads=4, embeddings=256, mlp_ratio=4,
                          dropout=0.1, attention=AttentionLayout.ALTERNATING_GLOBAL_STRUCTURED,
                          activation=Activation.TANH, position=PositionEncoding.ROPE,
                          rope_theta=10000.0, rope_fraction=1.0,
                          use_type_embeddings=True, concat_type_embeddings=False,
                          streams=StructuredStreams(inside_beat=True, beat_memory=True,
                                                    cycle_memory=True)),
        train=TrainConfig(epochs=10, batch_size=4, learning_rate=3e-4,
                          weight_decay=0.01, grad_clip=1.0, amp=True,
                          save_every_steps=200, log_every_steps=50,
                          early_stopping=False),
        evaluation=EvalConfig(batch_size=4, every_steps=200, save_plots=True),
    ),
    'no_subd_small': RunConfig(
        name='no_subd_small',
        data=DataConfig(variant=DatasetVariant.NO_SUBD, context_size=1024, stride=512),
    ),
}
SELECTED_CONFIG = 'with_subd_small'
config = RUN_CONFIGS[SELECTED_CONFIG]
print(json.dumps(config.to_dict(), indent=2))
print('Config hash:', config.fingerprint)


## 4. Prepare or reuse the exact dataset recipe

The package hashes all source files and relevant preparation settings. Changing context, stride, variant, decorated data or source bytes creates a different cache. Per-file cycle splitting remains 70/20/10; the test set includes later cycles of files also used in training.

In [ ]:
from ldm.experiment_workflow import prepare_or_load_dataset
dataset = prepare_or_load_dataset(config, ordinary, decorated, cache_root)
manifest = json.loads((dataset / 'dataset_manifest.json').read_text(encoding='utf-8'))
print(dataset)
print(json.dumps(manifest['stats'], indent=2))


## 5. Start a **new** run

Running this cell always creates a new unique Drive folder. It prints progress live and writes `training.log`, `last.pt`, `best.pt`, evaluation events and final metrics. It never resumes an old run.

In [ ]:
from ldm.experiment_workflow import start_run
run = start_run(config, dataset, runs_root)
print('New run ID:', run.name)


## 6. List and inspect runs

Set `SELECT_RUN_ID` to any run ID printed by the start cell. This cell does not modify training. The older fixed `runs/with_subd` folder can still be inspected manually using its `metrics.json` and checkpoints; it is not rewritten.

In [ ]:
from ldm.experiment_workflow import list_runs, inspect_run
for item in list_runs(runs_root):
    print(item['run_id'], item['status'], item.get('last_step'), item.get('best_val_loss'))
SELECT_RUN_ID = run.name if 'run' in globals() else None  # Or type an existing run ID.
if SELECT_RUN_ID:
    run = runs_root / SELECT_RUN_ID
    print(json.dumps(inspect_run(run), indent=2)[:12000])


## 7. Resume an existing named run

Run only when you explicitly want to continue `SELECT_RUN_ID`. The saved run config is read from Drive, so changing the currently selected preset does not alter the model. `TARGET_EPOCHS` is the new total, including completed epochs. A lower target is rejected.

In [ ]:
from ldm.experiment_workflow import resume_run
RESUME_NOW = False
TARGET_EPOCHS = 10
if RESUME_NOW:
    assert SELECT_RUN_ID, 'Select an existing run ID first'
    result = resume_run(runs_root / SELECT_RUN_ID, target_epochs=TARGET_EPOCHS)
    print('Stopped:', result['stop_reason'], 'at step', result['steps'])
else:
    print('Resume skipped. Set RESUME_NOW=True when ready.')


## 8. Metrics, plots, comparison

Validation selects `best.pt`; test metrics describe that selected checkpoint. The package scores each owned target once and reports combined, ordinary and decorated measures. Test cycles are from files also represented in training.

In [ ]:
from ldm.experiment_workflow import compare_runs
assert SELECT_RUN_ID, 'Select a run first'
run = runs_root / SELECT_RUN_ID
metrics = json.loads((run / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps({k: metrics[k] for k in ('steps', 'best_step', 'best_val_loss', 'test_loss', 'stop_reason')}, indent=2))
print('Test token accuracy:', metrics['test_report']['combined']['token_accuracy'])
print('Test HIT accuracy:', metrics['test_report']['combined']['hit_accuracy'])
print('Test skeleton HIT accuracy:', metrics['test_report']['combined']['skeleton_hit_accuracy'])
print(json.dumps(compare_runs([runs_root / item['run_id'] for item in list_runs(runs_root) if (runs_root / item['run_id'] / 'metrics.json').exists()]), indent=2))
from ldm.report_ldm import save_history_plot, save_comparison
if inspect_run(run)['config']['evaluation']['save_plots']:
    print('Plot:', save_history_plot(run))
selected_runs = [runs_root / item['run_id'] for item in list_runs(runs_root) if (runs_root / item['run_id'] / 'metrics.json').exists()]
if selected_runs:
    print('Comparison CSV:', save_comparison(selected_runs, storage_root / 'comparisons' / 'runs.csv'))


## 9. Build and validate a local Hugging Face snapshot

This exports `best.pt`. Snapshot validation loads AutoModel and AutoProcessor in an isolated process and checks a five-line round trip.

In [ ]:
import hashlib
variant = json.loads((run / 'run_config.json').read_text())['model_config']['dataset_variant']
best_checkpoint = run / 'best.pt'
checkpoint_sha256 = hashlib.sha256(best_checkpoint.read_bytes()).hexdigest()
snapshot = project / 'snapshots' / f'{variant}-{checkpoint_sha256[:12]}-{code_commit[:8]}'
release_manifest_path = snapshot / 'metadata' / 'release_manifest.json'
if snapshot.exists() and not release_manifest_path.exists():
    raise RuntimeError(f'Incomplete snapshot: {snapshot}')
if not snapshot.exists():
    subprocess.run([sys.executable, '-m', 'ldm.cli', 'snapshot', '--run', str(run), '--output', str(snapshot)], check=True)
release_manifest = json.loads(release_manifest_path.read_text(encoding='utf-8'))
assert release_manifest['source_checkpoint_sha256'] == checkpoint_sha256
subprocess.run([sys.executable, '-m', 'ldm.cli', 'validate', '--snapshot', str(snapshot)], check=True)
print('Validated:', snapshot)


## 10. Generate several reviews

Choose any valid five-line prompt files. Each review saves prompt, generated rhythm, settings and discarded-cycle report under Drive.

In [ ]:
from ldm.review_ldm import InferenceConfig, review_generation
from ldm.derbake_ldm import RhythmSequence, parse_derbake, write_derbake
source_file = sorted(ordinary.glob('*.derbake'))[0]
prompt_dir = storage_root / 'prompts' / SELECT_RUN_ID
prompt_dir.mkdir(parents=True, exist_ok=True)
prompt_file = prompt_dir / f'{source_file.stem}_first_cycle.derbake'
write_derbake(RhythmSequence((parse_derbake(source_file).rhythm.cycles[0],)), prompt_file)
REVIEW_PROMPTS = [prompt_file]  # Add more valid prompts here.

REVIEW_SEEDS = [1337, 2026]
for prompt in REVIEW_PROMPTS:
    for seed in REVIEW_SEEDS:
        report = review_generation(snapshot, prompt, storage_root / 'reviews' / SELECT_RUN_ID,
                                   InferenceConfig(num_cycles=1, seed=seed, max_new_tokens=2048))
        print(report['output'], report['generation'])


## 11. Optional WAV preview and embedding analysis

Audio needs four local WAV sample directories (`doums`, `taks`, `tiks`, `pa2s`). Leave `SAMPLE_DIRS` empty to skip. Decorated hits use their base sample. Embedding neighbors are exploratory.

In [ ]:
from ldm.review_ldm import render_wav, embedding_neighbors
SAMPLE_DIRS = {}  # e.g. {'doums': Path(...), 'taks': Path(...), 'tiks': Path(...), 'pa2s': Path(...)}
if SAMPLE_DIRS:
    generated_file = Path(report['output'])
    print(render_wav(generated_file, SAMPLE_DIRS, generated_file.with_suffix('.wav')))
else:
    print('Audio skipped: provide four sample directories to enable it.')
print(json.dumps(embedding_neighbors(snapshot, ['HIT_D', 'HIT_D_decorated'] +
    (['SUBD_1', 'SUBD_2'] if variant == 'with_subd' else [])), indent=2))

from ldm.review_ldm import contextual_projection
RUN_CONTEXTUAL_ANALYSIS = False  # Bounded optional CPU analysis.
if RUN_CONTEXTUAL_ANALYSIS:
    print(contextual_projection(snapshot, dataset, run / 'analysis'))


## 12. Publish the reviewed snapshot

Set the switch only after reviewing metrics and generated files. This uploads the selected snapshot to the matching HF model repository. Check that it represents a production run rather than a short pipeline check.

In [ ]:
from huggingface_hub import HfApi
from transformers import AutoModel, AutoProcessor
repo_id = {'with_subd': 'Appodoom/ldm-with-subd', 'no_subd': 'Appodoom/ldm-no-subd'}[variant]
api = HfApi()
remote_info = api.repo_info(repo_id=repo_id, repo_type='model')
remote_files = {item.rfilename for item in remote_info.siblings}
local_files = {path.relative_to(snapshot).as_posix() for path in snapshot.rglob('*') if path.is_file()}
stale_files = sorted(remote_files - local_files - {'.gitattributes'})
print('Remote files to remove on release:', stale_files)
PUBLISH_TO_HF = False
published_commit = None
if PUBLISH_TO_HF:
    assert metrics['stop_reason'] in {'completed', 'early_stopped'}
    assert release_manifest['source_checkpoint_sha256'] == hashlib.sha256(best_checkpoint.read_bytes()).hexdigest()
    subprocess.run([sys.executable, '-m', 'ldm.cli', 'validate', '--snapshot', str(snapshot)], check=True)
    uploaded = api.upload_folder(repo_id=repo_id, repo_type='model', folder_path=str(snapshot),
                                 commit_message=f'Release {variant} model',
                                 parent_commit=remote_info.sha, delete_patterns=stale_files)
    published_commit = uploaded.oid
    print(f'https://huggingface.co/{repo_id}/tree/{published_commit}')
else:
    print('Upload skipped.')


## 13. Verify the exact remote revision

Use the same pinned commit for AutoModel and AutoProcessor in downstream inference.

In [ ]:
if published_commit is None:
    print('Remote validation skipped.')
else:
    remote_snapshot = Path(snapshot_download(repo_id=repo_id, repo_type='model',
        revision=published_commit, local_dir=project / 'remote_validation' / variant / published_commit))
    subprocess.run([sys.executable, '-m', 'ldm.cli', 'validate', '--snapshot', str(remote_snapshot)], check=True)
    remote_model = AutoModel.from_pretrained(repo_id, revision=published_commit, trust_remote_code=True).eval()
    remote_processor = AutoProcessor.from_pretrained(repo_id, revision=published_commit, trust_remote_code=True)
    assert remote_model.config.vocab_hash == remote_processor.tokenizer.atomic.hash
    release_record = {'repo_id': repo_id, 'revision': published_commit, 'run_id': SELECT_RUN_ID,
        'code_commit': code_commit, 'checkpoint_sha256': checkpoint_sha256,
        'dataset_fingerprint': metrics['dataset_fingerprint'],
        'metric_version': metrics['test_report']['metric_version']}
    record_path = storage_root / 'releases' / f'{variant}-{published_commit[:12]}.json'
    record_path.parent.mkdir(parents=True, exist_ok=True)
    record_path.write_text(json.dumps(release_record, indent=2) + '\n', encoding='utf-8')
    print('Verified:', published_commit, '| Record:', record_path)
